**Purpose**

Generate embeddings for all chunked assets using GPU in Colab and persist them directly into Google Drive.

This notebook processes:

12 datasets
× 3 chunking methods
× 2 embedding models
= 72 embedding artifacts
=


In [19]:
# Install dependencies

!pip install -q \ sentence-transformers \ transformers \ accelerate \ pandas \ numpy \ tqdm

In [20]:
# Mount google Drive

from google.colab import drive
drive.mount('/content/drive')


import os

print(os.path.exists("/content/drive/MyDrive"))
print(os.listdir("/content/drive") if os.path.exists("/content/drive") else "Not mounted")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True
['.shortcut-targets-by-id', 'MyDrive', '.Trash-0', '.Encrypted']


In [21]:
"""
import os

print(os.getcwd())
print(os.listdir())
"""


import datasets
print(datasets)
print(getattr(datasets, "__file__", "No file"))

<module 'datasets' from '/usr/local/lib/python3.12/dist-packages/datasets/__init__.py'>
/usr/local/lib/python3.12/dist-packages/datasets/__init__.py


In [22]:
#Imports

import os
import gc
import json
import pickle
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer

import torch


In [23]:
#Verify GPU

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))
else:
  print("WARNING: GPU not detected.")

CUDA Available: True
GPU: Tesla T4


In [24]:
#Configuration
PROJECT_ROOT = "/content/drive/MyDrive/RAGBenchmark"
CHUNK_DIR = os.path.join(PROJECT_ROOT, "chunks")
EMBEDDING_DIR = os.path.join(PROJECT_ROOT, "embeddings")
os.makedirs(EMBEDDING_DIR, exist_ok=True)
EMBEDDING_MODELS = {
    "bge_base": "BAAI/bge-base-en-v1.5",
}
BATCH_SIZE = 64
NORMALIZE = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Reading chunks from :", CHUNK_DIR)
print("Saving embeddings to:", EMBEDDING_DIR)

Reading chunks from : /content/drive/MyDrive/RAGBenchmark/chunks
Saving embeddings to: /content/drive/MyDrive/RAGBenchmark/embeddings


In [25]:
# Verify Chunk count (Generated from 01_generate_chunks)

chunk_count = 0
for root, dirs, files in os.walk(CHUNK_DIR):
  for f in files:
     if f.endswith(".pkl"):
      chunk_count += 1
      print("Chunk files found:", chunk_count)

Chunk files found: 1
Chunk files found: 2
Chunk files found: 3
Chunk files found: 4
Chunk files found: 5
Chunk files found: 6
Chunk files found: 7
Chunk files found: 8
Chunk files found: 9
Chunk files found: 10
Chunk files found: 11
Chunk files found: 12
Chunk files found: 13
Chunk files found: 14
Chunk files found: 15
Chunk files found: 16
Chunk files found: 17
Chunk files found: 18
Chunk files found: 19
Chunk files found: 20
Chunk files found: 21
Chunk files found: 22
Chunk files found: 23
Chunk files found: 24
Chunk files found: 25
Chunk files found: 26
Chunk files found: 27
Chunk files found: 28
Chunk files found: 29
Chunk files found: 30
Chunk files found: 31
Chunk files found: 32
Chunk files found: 33
Chunk files found: 34
Chunk files found: 35
Chunk files found: 36


In [26]:
def flatten_chunks(chunk_data):
    flattened = []

    for record in chunk_data:

        question_id = str(record["id"])

        for chunk in record["chunks"]:

            if chunk is None:
                continue

            if str(chunk).strip() == "":
                continue

            flattened.append({
                "question_id": question_id,
                "chunk": str(chunk)
            })

    return flattened

In [27]:
#Generate Embeddings

summary = []
for embedding_name, model_name in EMBEDDING_MODELS.items():
   print("=" * 80)
   print("Loading Model:", model_name)

   model = SentenceTransformer(
        model_name,
        device=DEVICE
   )

for domain in os.listdir(CHUNK_DIR):

       domain_path = os.path.join(CHUNK_DIR, domain)

       if not os.path.isdir(domain_path):
           continue

       output_domain = os.path.join(
            EMBEDDING_DIR,
            domain
       )

       os.makedirs(output_domain, exist_ok=True)

       for file in os.listdir(domain_path):

        if not file.endswith(".pkl"):
           continue

        dataset_method = file.replace(
            ".pkl",
            ""
        )


        save_path = os.path.join(
          output_domain,
          f"{dataset_method}_{embedding_name}.npz"
        )

        if os.path.exists(save_path):
            print("Skipping:", os.path.basename(save_path))
            continue

        print("\n" + "=" * 60)
        print(dataset_method)

        chunk_path = os.path.join(
            domain_path,
            file
        )

        with open(chunk_path, "rb") as f:
          chunk_data = pickle.load(f)

        flattened = flatten_chunks(chunk_data)

        if len(flattened) == 0:
          print("No chunks found. Skipping.")
          continue

        texts = [
            item["chunk"]
            for item in flattened
            ]

        question_ids = [
            item["question_id"]
            for item in flattened
        ]

        embeddings = model.encode(
             texts,
             batch_size=BATCH_SIZE,
             show_progress_bar=True,
             convert_to_numpy=True,
             normalize_embeddings=NORMALIZE
        )

        save_path = os.path.join(
            output_domain,
            f"{dataset_method}_{embedding_name}.npz"
        )

        # Skip if already generated
        if os.path.exists(save_path):
            print("Skipping:", os.path.basename(save_path))
            continue

        np.savez_compressed(
            save_path,
            embeddings=embeddings,
            question_ids=np.array(question_ids),
            chunks=np.array(texts, dtype=object)

        )

        summary.append({
            "domain": domain,
            "dataset_method": dataset_method,
            "embedding": embedding_name,
            "vectors": embeddings.shape[0],
            "dimension": embeddings.shape[1],
            "path": save_path
        })
        print( f"Saved: {embeddings.shape}" )

        del chunk_data
        del flattened
        del texts
        del question_ids
        del embeddings

        gc.collect()

        if torch.cuda.is_available():
          torch.cuda.empty_cache()
del model
gc.collect()
if torch.cuda.is_available():
  torch.cuda.empty_cache()


Loading Model: BAAI/bge-base-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

Saved: (3, 768)

tatqa_metadata


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: (3, 768)

tatqa_small2big


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: (3, 768)


In [28]:
import os

possible_paths = [
    "/content/chunks",
    "/content/content/chunks",
    "/content/drive/MyDrive/RAGBenchmark/chunks"
]

for path in possible_paths:
    print("\n", path)
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        print("Contents:", os.listdir(path))


 /content/chunks
Exists: False

 /content/content/chunks
Exists: False

 /content/drive/MyDrive/RAGBenchmark/chunks
Exists: True
Contents: ['biomedical', 'general', 'legal', 'support', 'finance', 'chunk_summary.csv']


In [29]:
# Embedding Summary
summary_df = pd.DataFrame(summary)
summary_df

,domain,dataset_method,embedding,vectors,dimension,path
0,biomedical,covidqa_standard,bge_base,1,768,/content/drive/MyDrive/RAGBenchmark/embeddings...
1,biomedical,covidqa_metadata,bge_base,1,768,/content/drive/MyDrive/RAGBenchmark/embeddings...
2,biomedical,covidqa_small2big,bge_base,2,768,/content/drive/MyDrive/RAGBenchmark/embeddings...
3,biomedical,pubmedqa_standard,bge_base,1,768,/content/drive/MyDrive/RAGBenchmark/embeddings...
4,biomedical,pubmedqa_metadata,bge_base,1,768,/content/drive/MyDrive/RAGBenchmark/embeddings...
5,biomedical,pubmedqa_small2big,bge_base,1,768,/content/drive/MyDrive/RAGBenchmark/embeddings...
6,general,hotpotqa_standard,bge_base,1,768,/content/drive/MyDrive/RAGBenchmark/embeddings...
7,general,hotpotqa_metadata,bge_base,1,768,/content/drive/MyDrive/RAGBenchmark/embeddings...
8,general,hotpotqa_small2big,bge_base,1,768,/content/drive/MyDrive/RAGBenchmark/embeddings...
9,general,msmarco_standard,bge_base,2,768,/content/drive/MyDrive/RAGBenchmark/embeddings...


In [30]:
# Persist Summary

summary_path = os.path.join(
     EMBEDDING_DIR, "embedding_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

print("Summary saved:", summary_path)

Summary saved: /content/drive/MyDrive/RAGBenchmark/embeddings/embedding_summary.csv


In [31]:
#Validate Output Structure

for root, dirs, files in os.walk(EMBEDDING_DIR):
  level = root.replace(
      EMBEDDING_DIR,
      ""
  ).count(os.sep)

  indent = " " * 4 * level

  print(f"{indent}{os.path.basename(root)}/")
  for file in files:
    print(f"{indent} {file}")

embeddings/
 embedding_summary.csv
    biomedical/
     covidqa_standard_llm_embedder.npz
     covidqa_metadata_llm_embedder.npz
     covidqa_small2big_llm_embedder.npz
     pubmedqa_standard_llm_embedder.npz
     pubmedqa_metadata_llm_embedder.npz
     pubmedqa_small2big_llm_embedder.npz
     covidqa_standard_bge_base.npz
     covidqa_metadata_bge_base.npz
     covidqa_small2big_bge_base.npz
     pubmedqa_standard_bge_base.npz
     pubmedqa_metadata_bge_base.npz
     pubmedqa_small2big_bge_base.npz
    general/
     hotpotqa_standard_llm_embedder.npz
     hotpotqa_metadata_llm_embedder.npz
     hotpotqa_small2big_llm_embedder.npz
     msmarco_standard_llm_embedder.npz
     msmarco_metadata_llm_embedder.npz
     msmarco_small2big_llm_embedder.npz
     hagrid_standard_llm_embedder.npz
     hagrid_metadata_llm_embedder.npz
     hagrid_small2big_llm_embedder.npz
     expertqa_standard_llm_embedder.npz
     expertqa_metadata_llm_embedder.npz
     expertqa_small2big_llm_embedder.npz
     ho

In [32]:
# Verify Artifact Count

embedding_count = 0

for root, dirs, files in os.walk(EMBEDDING_DIR):
  for f in files:
    if f.endswith(".npz"):
      embedding_count += 1


print("Embedding files:", {embedding_count})

Embedding files: {72}


In [33]:
#Quick Validation

sample = summary_df.iloc[0]
data = np.load(
    sample["path"],
    allow_pickle=True
    )
print("Embeddings:", data["embeddings"].shape)
print("Question IDs:", data["question_ids"].shape)
print("Chunks:", data["chunks"].shape)

Embeddings: (1, 768)
Question IDs: (1,)
Chunks: (1,)


In [34]:
for domain in os.listdir(CHUNK_DIR):

    domain_path = os.path.join(CHUNK_DIR, domain)

    if not os.path.isdir(domain_path):
        continue

    ...